# 17 — P2: Full ViT MILP Verification + Correctness Tests

**Plan 2 — Phases 4.4 + 4.5 + 4.6 + 4.7.**

Compose the per-component MILP encoders (RMSNorm from notebook 15, MHSA from
notebook 16, plus a big-M ReLU MLP block here) into a single
`verify_vit_milp(model, x0, ε, y_true)` that produces a sound robustness
verdict.

## What's added in this notebook

1. **`encode_relu_bigM`** — exact big-M ReLU encoding (reuses Plan 1 idea).
2. **`encode_mlp_block`** — `Linear → ReLU → Linear`.
3. **`encode_transformer_block`** — `RMSNorm → MHSA → +x → RMSNorm → MLP → +x`.
4. **IBP for the full ViT** — propagates `[x − ε, x + ε] ∩ [0,1]` through
   patch-embed, pos-add, blocks, final norm, mean-pool, head. Returns
   per-component bounds the MILP encoder needs (PWL domains, McCormick boxes).
5. **`encode_vit`** — full forward encoding into a Gurobi model.
6. **`verify_vit_milp`** — for each `c ≠ y_true`, solve `min margin = z_y − z_c`;
   returns `verified` if all minimums ≥ 0, `falsified` if any negative
   (with concrete counterexample), or `inconclusive` on timeout.

## Phase 4.7 correctness tests
- **Identity at ε = 0** (10 samples): MILP `worst_margin` matches forward-pass
  margin within `pwl_error_bound` (tightest at ε=0, where input box collapses
  to a point and PWL becomes degenerate-but-bounded).
- **Counterexample validity**: every falsified counterexample is misclassified
  by the network. Buggy encodings produce phantom counterexamples.
- **Soundness vs IBP**: every IBP-verified sample is also MILP-verified.

## Compute budget
The full trained ViT-Tiny (`N=49, E=64, H=2, layers=2`) gives ≈10⁵ binary
variables; expect minutes per `(sample, c)` even with Gurobi. Tests in this
notebook use a **tiny custom ViT** (`img_size=8`, `embed_dim=4`, `H=1`,
`layers=1`) to verify the encoding semantics. The real trained model is
exercised in notebook 18 (hybrid verifier) where MILP is the *last resort*.


In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.environ['GRB_LICENSE_FILE'] = '/content/drive/My Drive/thesis-formal-verification/gurobi.lic'
os.environ['PATH'] = '/content/drive/My Drive/thesis-formal-verification:' + os.environ.get('PATH', '')

print('Google Drive mounted and Gurobi path added to os.environ')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and Gurobi path added to os.environ


In [2]:
import gurobipy as gp
from gurobipy import GRB
HAS_GUROBI = True
# Confirm the paid license is picked up — should NOT say "Restricted license":
gp.Model('lic_check').dispose()

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2819113
Academic license 2819113 - for non-commercial use only - registered to m____@ma.iitr.ac.in


In [3]:
!pip install -q numpy torch torchvision gurobipy

In [4]:
from __future__ import annotations
import math, json, time, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List, Tuple, Optional, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as transforms

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    HAS_GUROBI = False
    print(f'Gurobi not available ({e}); MILP tests skipped.')

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1234); np.random.seed(1234)
print(f'Device: {device}  Gurobi={HAS_GUROBI}')

Device: cuda  Gurobi=True


In [5]:
# ── ViT-Tiny architecture (matches notebooks 09–16) ──────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__(); self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape; H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        s = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        a = F.softmax(s, dim=-1)
        o = torch.matmul(a, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(o)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__(); h = embed_dim * mlp_ratio
        self.fc1 = nn.Linear(embed_dim, h); self.fc2 = nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); x = x + self.mlp(self.norm2(x)); return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
                                     for _ in range(num_layers)])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

In [6]:
# ── PWL primitives (verbatim from notebook 14) ──────────────────────────
@dataclass
class PWLBracket:
    name: str; breakpoints: List[float]
    slope_lo: List[float]; int_lo: List[float]
    slope_up: List[float]; int_up: List[float]
    @property
    def n_pieces(self): return len(self.breakpoints) - 1
    @property
    def domain(self):   return (self.breakpoints[0], self.breakpoints[-1])

def _line(p1, p2):
    s = (p2[1]-p1[1])/(p2[0]-p1[0]); return s, p1[1]-s*p1[0]
def _tan(f, df, x0):
    s = df(x0); return s, f(x0)-s*x0

def build_pwl_convex(f, df, a, b, n, name):
    bps = list(np.linspace(a, b, n+1))
    sl, il, su, iu = [], [], [], []
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        s_u, i_u = _line((x_l,f(x_l)),(x_r,f(x_r))); su.append(s_u); iu.append(i_u)
        s_l, i_l = _tan(f, df, 0.5*(x_l+x_r));       sl.append(s_l); il.append(i_l)
    return PWLBracket(name, bps, sl, il, su, iu)

def pwl_square(a, b, n=8):  return build_pwl_convex(lambda x: x*x, lambda x: 2*x, a, b, n, 'sq')
def pwl_exp(a, b, n=8):      return build_pwl_convex(math.exp, math.exp, a, b, n, 'exp')
def pwl_inv_pos(a, b, n=8):
    assert a > 0
    return build_pwl_convex(lambda x: 1.0/x, lambda x: -1.0/(x*x), a, b, n, 'inv')
def pwl_inv_sqrt_pos(a, b, n=8):
    assert a > 0
    return build_pwl_convex(lambda t: 1.0/math.sqrt(t),
                            lambda t: -0.5*t**(-1.5), a, b, n, 'isq')

def add_pwl_bracket(model, x_var, y_var, br: PWLBracket, big_M=None, prefix=''):
    n, bps = br.n_pieces, br.breakpoints
    a, b = bps[0], bps[-1]
    if big_M is None:
        scope = max(
            max(abs(br.slope_lo[k])*(b-a)+abs(br.int_lo[k]) for k in range(n)),
            max(abs(br.slope_up[k])*(b-a)+abs(br.int_up[k]) for k in range(n)),
        )
        big_M = max(2.0*scope, 1.0)
    delta = [model.addVar(vtype=GRB.BINARY, name=f'{prefix}d_{k}') for k in range(n)]
    model.addConstr(gp.quicksum(delta) == 1)
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        model.addConstr(x_var >= x_l - big_M*(1-delta[k]))
        model.addConstr(x_var <= x_r + big_M*(1-delta[k]))
        model.addConstr(y_var >= br.slope_lo[k]*x_var + br.int_lo[k] - big_M*(1-delta[k]))
        model.addConstr(y_var <= br.slope_up[k]*x_var + br.int_up[k] + big_M*(1-delta[k]))

def add_mccormick(model, a_var, b_var, a_l, a_u, b_l, b_u, prefix='', y_lb=None, y_ub=None):
    z_lo = min(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    z_up = max(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    if y_lb is not None: z_lo = max(z_lo, y_lb)
    if y_ub is not None: z_up = min(z_up, y_ub)
    z = model.addVar(lb=z_lo, ub=z_up, name=f'{prefix}z')
    model.addConstr(z >= a_l*b_var + b_l*a_var - a_l*b_l)
    model.addConstr(z >= a_u*b_var + b_u*a_var - a_u*b_u)
    model.addConstr(z <= a_u*b_var + b_l*a_var - a_u*b_l)
    model.addConstr(z <= a_l*b_var + b_u*a_var - a_l*b_u)
    return z

In [7]:
# ── Big-M ReLU encoder (Plan 1 reuse) ───────────────────────────────────
def add_relu_bigM(model, x_var, x_lo, x_up, prefix=''):
    """y = max(0, x_var) where x_var ∈ [x_lo, x_up]. Returns y var."""
    if x_lo >= 0:
        # always active: y = x
        y = model.addVar(lb=x_lo, ub=x_up, name=f'{prefix}y')
        model.addConstr(y == x_var); return y
    if x_up <= 0:
        y = model.addVar(lb=0.0, ub=0.0, name=f'{prefix}y')
        model.addConstr(y == 0.0); return y
    # mixed: big-M
    delta = model.addVar(vtype=GRB.BINARY, name=f'{prefix}a')
    y = model.addVar(lb=0.0, ub=x_up, name=f'{prefix}y')
    model.addConstr(y >= x_var)
    model.addConstr(y <= x_up * delta)
    model.addConstr(y <= x_var - x_lo * (1 - delta))
    return y

# ── IBP helpers (numpy) ─────────────────────────────────────────────────
def ibp_lin(x_l, x_u, W, b):
    Wp = np.maximum(W, 0); Wn = np.minimum(W, 0)
    y_l = x_l @ Wp.T + x_u @ Wn.T
    y_u = x_u @ Wp.T + x_l @ Wn.T
    if b is not None:
        y_l = y_l + b; y_u = y_u + b
    return y_l, y_u

def ibp_relu(x_l, x_u): return np.maximum(x_l, 0), np.maximum(x_u, 0)

def ibp_rmsnorm(x_l, x_u, gamma, eps):
    """per-token RMSNorm IBP. x_l/x_u: (..., D)."""
    sq_lo = np.where((x_l <= 0) & (x_u >= 0), 0.0, np.minimum(x_l**2, x_u**2))
    sq_up = np.maximum(x_l**2, x_u**2)
    msq_l = sq_lo.mean(axis=-1, keepdims=True)
    msq_u = sq_up.mean(axis=-1, keepdims=True)
    inv_l = 1.0 / np.sqrt(msq_u + eps)
    inv_u = 1.0 / np.sqrt(np.maximum(msq_l, 0) + eps)
    # y = gamma * x * inv  → element-wise interval product
    inv_l_b = np.broadcast_to(inv_l, x_l.shape)
    inv_u_b = np.broadcast_to(inv_u, x_u.shape)
    c1 = x_l*inv_l_b; c2 = x_l*inv_u_b; c3 = x_u*inv_l_b; c4 = x_u*inv_u_b
    z_l = np.minimum(np.minimum(c1,c2), np.minimum(c3,c4))
    z_u = np.maximum(np.maximum(c1,c2), np.maximum(c3,c4))
    gp_, gn_ = np.maximum(gamma, 0), np.minimum(gamma, 0)
    y_l = z_l*gp_ + z_u*gn_
    y_u = z_u*gp_ + z_l*gn_
    return y_l, y_u

def ibp_attention_bounds(x_lo, x_up, attn):
    N, E = x_lo.shape; H, D = attn.num_heads, attn.head_dim
    scale = attn.scale
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy() if attn.W_v.bias is not None else None
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy() if attn.W_o.bias is not None else None
    Q_l, Q_u = ibp_lin(x_lo, x_up, Wq, None)
    K_l, K_u = ibp_lin(x_lo, x_up, Wk, None)
    V_l, V_u = ibp_lin(x_lo, x_up, Wv, bv)
    def to_h(a): return a.reshape(N, H, D).transpose(1, 0, 2)
    Q_l, Q_u = to_h(Q_l), to_h(Q_u)
    K_l, K_u = to_h(K_l), to_h(K_u)
    V_l, V_u = to_h(V_l), to_h(V_u)
    Q_e_l = Q_l[:, :, None, :]; Q_e_u = Q_u[:, :, None, :]
    K_e_l = K_l[:, None, :, :]; K_e_u = K_u[:, None, :, :]
    c1 = Q_e_l*K_e_l; c2 = Q_e_l*K_e_u; c3 = Q_e_u*K_e_l; c4 = Q_e_u*K_e_u
    P_l = np.minimum(np.minimum(c1,c2), np.minimum(c3,c4))
    P_u = np.maximum(np.maximum(c1,c2), np.maximum(c3,c4))
    S_l = scale * P_l.sum(-1); S_u = scale * P_u.sum(-1)
    shift = S_u.max(axis=-1, keepdims=True)
    Sh_l = S_l - shift; Sh_u = S_u - shift
    E_l = np.exp(Sh_l); E_u = np.exp(Sh_u)
    SumE_l = E_l.sum(-1, keepdims=True); SumE_u = E_u.sum(-1, keepdims=True)
    SumE_l = np.maximum(SumE_l, 1e-9)
    Inv_l = 1.0/SumE_u; Inv_u = 1.0/SumE_l
    Inv_b_l = np.broadcast_to(Inv_l, E_l.shape); Inv_b_u = np.broadcast_to(Inv_u, E_u.shape)
    cA1 = E_l*Inv_b_l; cA2 = E_l*Inv_b_u; cA3 = E_u*Inv_b_l; cA4 = E_u*Inv_b_u
    A_l = np.maximum(np.minimum(np.minimum(cA1,cA2),np.minimum(cA3,cA4)), 0.0)
    A_u = np.minimum(np.maximum(np.maximum(cA1,cA2),np.maximum(cA3,cA4)), 1.0)
    A_e_l = A_l[..., None]; A_e_u = A_u[..., None]
    V_e_l = V_l[:, None, :, :]; V_e_u = V_u[:, None, :, :]
    cO1 = A_e_l*V_e_l; cO2 = A_e_l*V_e_u; cO3 = A_e_u*V_e_l; cO4 = A_e_u*V_e_u
    OP_l = np.minimum(np.minimum(cO1,cO2),np.minimum(cO3,cO4))
    OP_u = np.maximum(np.maximum(cO1,cO2),np.maximum(cO3,cO4))
    O_l = OP_l.sum(2); O_u = OP_u.sum(2)
    O_cat_l = O_l.transpose(1,0,2).reshape(N, E)
    O_cat_u = O_u.transpose(1,0,2).reshape(N, E)
    out_l, out_u = ibp_lin(O_cat_l, O_cat_u, Wo, bo)
    return dict(Q=(Q_l,Q_u),K=(K_l,K_u),V=(V_l,V_u),S=(S_l,S_u),shift=shift,
                S_shifted=(Sh_l,Sh_u),E=(E_l,E_u),SumE=(SumE_l,SumE_u),
                Inv=(Inv_l,Inv_u),A=(A_l,A_u),O=(O_l,O_u),out=(out_l,out_u))

def ibp_mlp_block(x_l, x_u, mlp):
    W1 = mlp.fc1.weight.detach().cpu().numpy(); b1 = mlp.fc1.bias.detach().cpu().numpy()
    W2 = mlp.fc2.weight.detach().cpu().numpy(); b2 = mlp.fc2.bias.detach().cpu().numpy()
    h_l, h_u = ibp_lin(x_l, x_u, W1, b1)
    r_l, r_u = ibp_relu(h_l, h_u)
    o_l, o_u = ibp_lin(r_l, r_u, W2, b2)
    return o_l, o_u, (h_l, h_u)   # h_l/h_u are pre-ReLU bounds, needed by big-M ReLU

def ibp_block(x_l, x_u, blk):
    g1 = blk.norm1.weight.detach().cpu().numpy(); eps = blk.norm1.eps
    n1_l, n1_u = ibp_rmsnorm(x_l, x_u, g1, eps)
    ibp_attn  = ibp_attention_bounds(n1_l, n1_u, blk.attn)
    a_l, a_u  = ibp_attn['out']
    r1_l, r1_u = x_l + a_l, x_u + a_u
    g2 = blk.norm2.weight.detach().cpu().numpy()
    n2_l, n2_u = ibp_rmsnorm(r1_l, r1_u, g2, eps)
    m_l, m_u, mlp_pre = ibp_mlp_block(n2_l, n2_u, blk.mlp)
    r2_l, r2_u = r1_l + m_l, r1_u + m_u
    return (r2_l, r2_u), dict(n1=(n1_l,n1_u), attn=ibp_attn, r1=(r1_l,r1_u),
                              n2=(n2_l,n2_u), mlp_pre=mlp_pre, m=(m_l,m_u))

def ibp_patch_embed(img_l, img_u, conv):
    Wp = conv.weight.clamp(min=0).detach().cpu().numpy()
    Wn = conv.weight.clamp(max=0).detach().cpu().numpy()
    b  = conv.bias.detach().cpu().numpy() if conv.bias is not None else None
    # use scipy-free conv via torch on numpy bridge
    Wp_t = torch.from_numpy(Wp); Wn_t = torch.from_numpy(Wn)
    img_l_t = torch.from_numpy(img_l).float(); img_u_t = torch.from_numpy(img_u).float()
    s, p = conv.stride, conv.padding
    l = F.conv2d(img_l_t, Wp_t.float(), None, s, p) + F.conv2d(img_u_t, Wn_t.float(), None, s, p)
    u = F.conv2d(img_u_t, Wp_t.float(), None, s, p) + F.conv2d(img_l_t, Wn_t.float(), None, s, p)
    if b is not None:
        b_t = torch.from_numpy(b).view(1, -1, 1, 1).float()
        l = l + b_t; u = u + b_t
    l = l.flatten(2).transpose(1, 2).numpy()  # (B, N, E)
    u = u.flatten(2).transpose(1, 2).numpy()
    return l, u

def ibp_vit(model, img_lo, img_up):
    """img_lo, img_up: (1, C, H, W). Returns logit bounds (1, num_classes) + per-stage dict."""
    pe_l, pe_u = ibp_patch_embed(img_lo, img_up, model.patch_embed.proj)
    pos = model.pos_embed.detach().cpu().numpy()  # (1, N, E)
    x_l = pe_l + pos; x_u = pe_u + pos
    stages = []
    for i, blk in enumerate(model.blocks):
        (x_l_new, x_u_new), info = ibp_block(x_l[0], x_u[0], blk)
        stages.append(dict(input=(x_l[0], x_u[0]), info=info, output=(x_l_new, x_u_new)))
        x_l = x_l_new[None]; x_u = x_u_new[None]
    g = model.norm.weight.detach().cpu().numpy(); eps = model.norm.eps
    nf_l, nf_u = ibp_rmsnorm(x_l[0], x_u[0], g, eps)
    pool_l = nf_l.mean(0); pool_u = nf_u.mean(0)
    Wh = model.head.weight.detach().cpu().numpy(); bh = model.head.bias.detach().cpu().numpy()
    log_l, log_u = ibp_lin(pool_l[None], pool_u[None], Wh, bh)
    return log_l, log_u, dict(stages=stages, norm_final=(nf_l, nf_u),
                              pool=(pool_l, pool_u), logits=(log_l[0], log_u[0]))

In [8]:
# ── encode_rmsnorm (notebook 15) ────────────────────────────────────────
def encode_rmsnorm(milp, x_vars, x_lo, x_up, gamma, eps_rms, n_pieces=8, prefix=''):
    D = len(x_vars)
    x_lo = np.asarray(x_lo, dtype=float); x_up = np.asarray(x_up, dtype=float)
    gamma = np.asarray(gamma, dtype=float)
    xsq_vars = []; xsq_lo_arr = np.zeros(D); xsq_up_arr = np.zeros(D)
    for i in range(D):
        a, b = float(x_lo[i]), float(x_up[i])
        if abs(a-b) < 1e-12:
            v = milp.addVar(lb=a*a, ub=a*a, name=f'{prefix}xsq_{i}')
            milp.addConstr(v == a*a); xsq_lo_arr[i]=xsq_up_arr[i]=a*a
            xsq_vars.append(v); continue
        br = pwl_square(a, b, n=n_pieces)
        if a <= 0 <= b: xsq_lo_arr[i] = 0.0
        else: xsq_lo_arr[i] = min(a*a, b*b)
        xsq_up_arr[i] = max(a*a, b*b)
        v = milp.addVar(lb=xsq_lo_arr[i]-1e-3, ub=xsq_up_arr[i]+1e-3,
                        name=f'{prefix}xsq_{i}')
        add_pwl_bracket(milp, x_vars[i], v, br, prefix=f'{prefix}xsq_{i}_')
        xsq_vars.append(v)
    mlo = float(xsq_lo_arr.mean()); mup = float(xsq_up_arr.mean())
    mean_xsq = milp.addVar(lb=mlo, ub=mup, name=f'{prefix}msq')
    milp.addConstr(mean_xsq == gp.quicksum(xsq_vars) / D)
    dlo, dup = mlo + eps_rms, mup + eps_rms
    denom = milp.addVar(lb=dlo, ub=dup, name=f'{prefix}den')
    milp.addConstr(denom == mean_xsq + eps_rms)
    if dlo <= 0: raise ValueError('eps_rms too small')
    br_inv = pwl_inv_sqrt_pos(dlo, dup, n=n_pieces)
    inv_lo, inv_up = 1.0/math.sqrt(dup), 1.0/math.sqrt(dlo)
    inv_rms = milp.addVar(lb=inv_lo-1e-3, ub=inv_up+1e-3, name=f'{prefix}inv')
    add_pwl_bracket(milp, denom, inv_rms, br_inv, prefix=f'{prefix}inv_')
    y_vars = []
    for i in range(D):
        a_l, a_u = float(x_lo[i]), float(x_up[i])
        # Compute z = x_i · inv_rms bounds explicitly. We cannot read z.LB/z.UB
        # right after addVar() — Gurobi defers attribute access until model.update().
        zlo = min(a_l*inv_lo, a_l*inv_up, a_u*inv_lo, a_u*inv_up)
        zup = max(a_l*inv_lo, a_l*inv_up, a_u*inv_lo, a_u*inv_up)
        if abs(a_l-a_u) < 1e-12:
            z = milp.addVar(lb=zlo, ub=zup, name=f'{prefix}xi_{i}')
            milp.addConstr(z == a_l * inv_rms)
        else:
            z = add_mccormick(milp, x_vars[i], inv_rms, a_l, a_u, inv_lo, inv_up,
                              prefix=f'{prefix}xi_{i}_')
        g = float(gamma[i])
        ylo = g*zlo if g>=0 else g*zup; yup = g*zup if g>=0 else g*zlo
        y = milp.addVar(lb=ylo, ub=yup, name=f'{prefix}y_{i}')
        milp.addConstr(y == g * z); y_vars.append(y)
    return y_vars

def encode_mhsa(milp, x_vars, x_lo, x_up, attn, ibp_b, n_pieces=8, prefix=''):
    N = len(x_vars); E = len(x_vars[0])
    H, D = attn.num_heads, attn.head_dim; scale = attn.scale
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy() if attn.W_v.bias is not None else np.zeros(E)
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy() if attn.W_o.bias is not None else np.zeros(E)
    Q_l,Q_u = ibp_b['Q']; K_l,K_u = ibp_b['K']; V_l,V_u = ibp_b['V']
    S_l,S_u = ibp_b['S']; shift = ibp_b['shift']
    Sh_l,Sh_u = ibp_b['S_shifted']; E_l_b,E_u_b = ibp_b['E']
    SumE_l,SumE_u = ibp_b['SumE']; Inv_l_b,Inv_u_b = ibp_b['Inv']
    A_l_b,A_u_b = ibp_b['A']

    def linproj(W, b_arr, p):
        E_out = W.shape[0]; out = []
        for i in range(N):
            row = []
            for e in range(E_out):
                expr = gp.quicksum(W[e, ep]*x_vars[i][ep] for ep in range(E)) + float(b_arr[e])
                v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{p}_{i}_{e}')
                milp.addConstr(v == expr); row.append(v)
            out.append(row)
        return out
    Qv = linproj(Wq, np.zeros(E), f'{prefix}Q')
    Kv = linproj(Wk, np.zeros(E), f'{prefix}K')
    Vv = linproj(Wv, bv,           f'{prefix}V')
    def to_h(v): return [[[v[i][h*D + d] for d in range(D)] for i in range(N)] for h in range(H)]
    Qh, Kh, Vh = to_h(Qv), to_h(Kv), to_h(Vv)

    Sv = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for j in range(N):
                z_terms = []
                for d in range(D):
                    z = add_mccormick(milp, Qh[h][i][d], Kh[h][j][d],
                        float(Q_l[h,i,d]), float(Q_u[h,i,d]),
                        float(K_l[h,j,d]), float(K_u[h,j,d]),
                        prefix=f'{prefix}qk_{h}_{i}_{j}_{d}_')
                    z_terms.append(z)
                s = milp.addVar(lb=float(S_l[h,i,j])-1e-3, ub=float(S_u[h,i,j])+1e-3,
                                name=f'{prefix}S_{h}_{i}_{j}')
                milp.addConstr(s == scale*gp.quicksum(z_terms))
                Sv[h][i][j] = s

    Av = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            sh_const = float(shift[h,i,0])
            E_row = []
            for j in range(N):
                shl, shu = float(Sh_l[h,i,j]), float(Sh_u[h,i,j])
                if shu - shl < 1e-9:
                    ec = math.exp(0.5*(shl+shu))
                    e = milp.addVar(lb=ec-1e-9, ub=ec+1e-9, name=f'{prefix}E_{h}_{i}_{j}')
                    milp.addConstr(e == ec)
                else:
                    br = pwl_exp(shl, shu, n=n_pieces)
                    ssh = milp.addVar(lb=shl, ub=shu, name=f'{prefix}Ssh_{h}_{i}_{j}')
                    milp.addConstr(ssh == Sv[h][i][j] - sh_const)
                    e = milp.addVar(lb=float(E_l_b[h,i,j])-1e-6, ub=float(E_u_b[h,i,j])+1e-6,
                                    name=f'{prefix}E_{h}_{i}_{j}')
                    add_pwl_bracket(milp, ssh, e, br, prefix=f'{prefix}E_{h}_{i}_{j}_')
                E_row.append(e)
            sel, seu = float(SumE_l[h,i,0]), float(SumE_u[h,i,0])
            sum_e = milp.addVar(lb=max(sel,1e-9), ub=seu, name=f'{prefix}SE_{h}_{i}')
            milp.addConstr(sum_e == gp.quicksum(E_row))
            il, iu = float(Inv_l_b[h,i,0]), float(Inv_u_b[h,i,0])
            if seu - sel < 1e-9:
                ic = 1.0 / (0.5*(sel+seu))
                inv_e = milp.addVar(lb=ic-1e-9, ub=ic+1e-9, name=f'{prefix}I_{h}_{i}')
                milp.addConstr(inv_e == ic)
            else:
                br_i = pwl_inv_pos(max(sel,1e-9), seu, n=n_pieces)
                inv_e = milp.addVar(lb=il-1e-6, ub=iu+1e-6, name=f'{prefix}I_{h}_{i}')
                add_pwl_bracket(milp, sum_e, inv_e, br_i, prefix=f'{prefix}I_{h}_{i}_')
            for j in range(N):
                a = add_mccormick(milp, E_row[j], inv_e,
                    float(E_l_b[h,i,j]), float(E_u_b[h,i,j]), il, iu,
                    prefix=f'{prefix}A_{h}_{i}_{j}_', y_lb=0.0, y_ub=1.0)
                Av[h][i][j] = a

    Ov = [[[None]*D for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for d in range(D):
                z_terms = []
                for j in range(N):
                    z = add_mccormick(milp, Av[h][i][j], Vh[h][j][d],
                        float(A_l_b[h,i,j]), float(A_u_b[h,i,j]),
                        float(V_l[h,j,d]), float(V_u[h,j,d]),
                        prefix=f'{prefix}av_{h}_{i}_{j}_{d}_')
                    z_terms.append(z)
                o = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}O_{h}_{i}_{d}')
                milp.addConstr(o == gp.quicksum(z_terms))
                Ov[h][i][d] = o
    O_cat = [[Ov[h][i][d] for h in range(H) for d in range(D)] for i in range(N)]
    out_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            expr = gp.quicksum(Wo[e, ep]*O_cat[i][ep] for ep in range(E)) + float(bo[e])
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}out_{i}_{e}')
            milp.addConstr(v == expr); row.append(v)
        out_vars.append(row)
    return out_vars

In [9]:
# ── encode_mlp_block ────────────────────────────────────────────────────
def encode_mlp_block(milp, x_vars, x_lo, x_up, mlp, ibp_pre, prefix=''):
    """Linear → ReLU → Linear (no residual; residual added by encode_block)."""
    N = len(x_vars); E = len(x_vars[0])
    W1 = mlp.fc1.weight.detach().cpu().numpy(); b1 = mlp.fc1.bias.detach().cpu().numpy()
    W2 = mlp.fc2.weight.detach().cpu().numpy(); b2 = mlp.fc2.bias.detach().cpu().numpy()
    Eh = W1.shape[0]
    h_l_arr, h_u_arr = ibp_pre  # pre-ReLU bounds (N, Eh)
    out_vars = []
    for i in range(N):
        # pre-ReLU h_e
        h_vars = []
        for e in range(Eh):
            expr = gp.quicksum(W1[e, ep]*x_vars[i][ep] for ep in range(E)) + float(b1[e])
            v = milp.addVar(lb=float(h_l_arr[i,e])-1e-3, ub=float(h_u_arr[i,e])+1e-3,
                            name=f'{prefix}h_{i}_{e}')
            milp.addConstr(v == expr); h_vars.append(v)
        # ReLU
        r_vars = [add_relu_bigM(milp, h_vars[e], float(h_l_arr[i,e]), float(h_u_arr[i,e]),
                                prefix=f'{prefix}r_{i}_{e}_') for e in range(Eh)]
        # fc2
        row = []
        for e in range(E):
            expr = gp.quicksum(W2[e, ep]*r_vars[ep] for ep in range(Eh)) + float(b2[e])
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}o_{i}_{e}')
            milp.addConstr(v == expr); row.append(v)
        out_vars.append(row)
    return out_vars

# ── encode_transformer_block ───────────────────────────────────────────
def encode_block(milp, x_vars, x_lo, x_up, blk, ibp_b, n_pieces=8, prefix=''):
    """Pre-LN: x = x + MHSA(N1(x)); x = x + MLP(N2(x))."""
    N = len(x_vars); E = len(x_vars[0])
    g1 = blk.norm1.weight.detach().cpu().numpy(); eps = blk.norm1.eps
    n1_l, n1_u = ibp_b['n1']
    # encode RMSNorm per-token
    n1_vars = []
    for i in range(N):
        ys = encode_rmsnorm(milp, x_vars[i], x_lo[i], x_up[i], g1, eps,
                            n_pieces=n_pieces, prefix=f'{prefix}n1_{i}_')
        n1_vars.append(ys)
    # MHSA
    a_vars = encode_mhsa(milp, n1_vars, n1_l, n1_u, blk.attn, ibp_b['attn'],
                         n_pieces=n_pieces, prefix=f'{prefix}atn_')
    # residual r1 = x + a
    r1_l, r1_u = ibp_b['r1']
    r1_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=float(r1_l[i,e]), ub=float(r1_u[i,e]),
                            name=f'{prefix}r1_{i}_{e}')
            milp.addConstr(v == x_vars[i][e] + a_vars[i][e]); row.append(v)
        r1_vars.append(row)
    # n2
    g2 = blk.norm2.weight.detach().cpu().numpy()
    n2_l, n2_u = ibp_b['n2']
    n2_vars = []
    for i in range(N):
        ys = encode_rmsnorm(milp, r1_vars[i], r1_l[i], r1_u[i], g2, eps,
                            n_pieces=n_pieces, prefix=f'{prefix}n2_{i}_')
        n2_vars.append(ys)
    # MLP block
    m_vars = encode_mlp_block(milp, n2_vars, n2_l, n2_u, blk.mlp,
                               ibp_b['mlp_pre'], prefix=f'{prefix}mlp_')
    # residual r2 = r1 + m
    r2_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}r2_{i}_{e}')
            milp.addConstr(v == r1_vars[i][e] + m_vars[i][e]); row.append(v)
        r2_vars.append(row)
    return r2_vars

# ── encode_vit ─────────────────────────────────────────────────────────
def encode_vit(milp, img_vars, img_lo, img_up, model, n_pieces=8, prefix=''):
    """
    img_vars: shape (C, H, W) of Gurobi vars, img_lo/img_up: numpy (1, C, H, W).
    Returns list of 10 logit vars.
    """
    log_l, log_u, ibp_full = ibp_vit(model, img_lo, img_up)
    # 1. Patch embed via Conv2d.  We materialise each output channel-position
    # as an explicit linear constraint over the corresponding input patch.
    conv = model.patch_embed.proj
    Wc = conv.weight.detach().cpu().numpy()    # (E, C, P, P)
    bc = conv.bias.detach().cpu().numpy() if conv.bias is not None else np.zeros(Wc.shape[0])
    C, Himg, Wimg = img_vars.shape
    P = conv.kernel_size[0]; H_out = Himg // P; W_out = Wimg // P; N = H_out * W_out
    E = Wc.shape[0]
    pe_vars = []
    for i_h in range(H_out):
        for i_w in range(W_out):
            row = []
            for e in range(E):
                terms = []
                for c_ in range(C):
                    for p_h in range(P):
                        for p_w in range(P):
                            w_ = float(Wc[e, c_, p_h, p_w])
                            if w_ != 0.0:
                                terms.append(w_ * img_vars[c_, i_h*P + p_h, i_w*P + p_w])
                expr = gp.quicksum(terms) + float(bc[e])
                v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY,
                                name=f'{prefix}pe_{i_h}_{i_w}_{e}')
                milp.addConstr(v == expr); row.append(v)
            pe_vars.append(row)
    # add positional embedding (constant)
    pos = model.pos_embed.detach().cpu().numpy()[0]  # (N, E)
    x_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}xi_{i}_{e}')
            milp.addConstr(v == pe_vars[i][e] + float(pos[i, e]))
            row.append(v)
        x_vars.append(row)
    # initial bounds for blocks
    pe_l, pe_u = ibp_patch_embed(img_lo, img_up, conv)
    x_l = pe_l[0] + pos; x_u = pe_u[0] + pos
    # 2. Blocks
    for bi, blk in enumerate(model.blocks):
        ibp_b = ibp_full['stages'][bi]['info']
        x_vars = encode_block(milp, x_vars, x_l, x_u, blk, ibp_b,
                               n_pieces=n_pieces, prefix=f'{prefix}b{bi}_')
        x_l, x_u = ibp_full['stages'][bi]['output']
    # 3. final RMSNorm
    g = model.norm.weight.detach().cpu().numpy(); eps = model.norm.eps
    nf_l, nf_u = ibp_full['norm_final']
    nf_vars = []
    for i in range(N):
        ys = encode_rmsnorm(milp, x_vars[i], x_l[i], x_u[i], g, eps,
                            n_pieces=n_pieces, prefix=f'{prefix}nf_{i}_')
        nf_vars.append(ys)
    # 4. mean-pool tokens
    pool_l, pool_u = ibp_full['pool']
    pool_vars = []
    for e in range(E):
        v = milp.addVar(lb=float(pool_l[e])-1e-3, ub=float(pool_u[e])+1e-3,
                        name=f'{prefix}pool_{e}')
        milp.addConstr(v == gp.quicksum(nf_vars[i][e] for i in range(N)) / N)
        pool_vars.append(v)
    # 5. head
    Wh = model.head.weight.detach().cpu().numpy(); bh = model.head.bias.detach().cpu().numpy()
    K = Wh.shape[0]
    log_vars = []
    for k in range(K):
        v = milp.addVar(lb=float(log_l[0,k])-1e-3, ub=float(log_u[0,k])+1e-3,
                        name=f'{prefix}log_{k}')
        milp.addConstr(v == gp.quicksum(float(Wh[k,e])*pool_vars[e] for e in range(E)) + float(bh[k]))
        log_vars.append(v)
    return log_vars, ibp_full

In [10]:
# ── verify_vit_milp ────────────────────────────────────────────────────
def verify_vit_milp(model: nn.Module, x0: np.ndarray, eps: float, true_label: int,
                    n_pieces: int = 8, time_limit: float = 300.0,
                    target_class: Optional[int] = None) -> Dict[str, Any]:
    """
    For each c ≠ true_label (or just target_class if given), solve
        min  z[true_label] - z[c]
        s.t. encoded forward pass; img ∈ [x0-eps, x0+eps] ∩ [0, 1]
    If any minimum < 0 → falsified; if all ≥ 0 → verified.
    Returns dict with {status, worst_margin, counterexample, time_per_class, n_pieces}.
    """
    if not HAS_GUROBI:
        return dict(status='skipped', reason='no gurobi')
    img_lo = np.clip(x0 - eps, 0.0, 1.0)
    img_up = np.clip(x0 + eps, 0.0, 1.0)
    targets = [target_class] if target_class is not None \
              else [c for c in range(10) if c != true_label]

    m = gp.Model('vit_verify')
    m.setParam('OutputFlag', 0); m.setParam('TimeLimit', time_limit)
    C, Himg, Wimg = x0.shape
    img_vars = np.empty((C, Himg, Wimg), dtype=object)
    for c in range(C):
        for h in range(Himg):
            for w in range(Wimg):
                img_vars[c, h, w] = m.addVar(lb=float(img_lo[c, h, w]),
                                             ub=float(img_up[c, h, w]),
                                             name=f'img_{c}_{h}_{w}')
    log_vars, ibp_full = encode_vit(
        m, img_vars, img_lo[None], img_up[None], model, n_pieces=n_pieces)
    m.update()

    times = {}; min_margins = {}
    counterexample = None; worst = float('inf'); worst_c = None
    for c in targets:
        margin_var = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'mar_{c}')
        m.addConstr(margin_var == log_vars[true_label] - log_vars[c])
        m.setObjective(margin_var, GRB.MINIMIZE)
        t0 = time.time(); m.optimize(); times[c] = time.time() - t0
        st = m.Status
        if st == GRB.INFEASIBLE:
            return dict(status='error', reason='infeasible')
        if st in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            margin = margin_var.X
        elif st == GRB.TIME_LIMIT and m.SolCount > 0:
            margin = margin_var.X
        else:
            return dict(status='inconclusive', reason=f'status={st}', time_per_class=times)
        min_margins[c] = margin
        if margin < worst:
            worst = margin; worst_c = c
            if margin < 0:
                ce = np.array([[[img_vars[ci,hi,wi].X for wi in range(Wimg)]
                                for hi in range(Himg)] for ci in range(C)])
                counterexample = ce
        # remove the margin constraint for the next target
        m.remove(margin_var)
        m.update()
        # Early-exit on falsification
        if margin < 0:
            return dict(status='falsified', worst_margin=margin, worst_class=worst_c,
                        counterexample=counterexample, time_per_class=times,
                        n_pieces=n_pieces)
    return dict(status='verified', worst_margin=worst, worst_class=worst_c,
                time_per_class=times, n_pieces=n_pieces)

In [11]:
# ── Tiny ViT for tractable correctness tests ────────────────────────────
TINY_CFG = dict(img_size=8, patch_size=4, in_channels=1, num_classes=3,
                embed_dim=4, num_heads=1, num_layers=1, mlp_ratio=2,
                eps_rms=1e-4)   # larger eps_rms keeps PWL inv_sqrt range bounded

def make_tiny_vit(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    m = ViTTiny(**TINY_CFG)
    # initialise small-ish weights to get well-behaved scores
    with torch.no_grad():
        for p in m.parameters(): p.mul_(0.5)
    m.eval(); return m

def make_tiny_input(seed=0):
    g = np.random.default_rng(seed)
    return g.uniform(0, 1, size=(1, TINY_CFG['img_size'], TINY_CFG['img_size'])).astype(np.float32)

In [12]:
# ── Phase 4.7 Test 1: identity at ε = 0 ─────────────────────────────────
def test_identity_at_eps0(n_samples=4, n_pieces=8):
    """
    With ε=0 the input box collapses to a point.  The MILP worst_margin
    should match the true forward-pass margin, modulo PWL approximation
    error (which is non-zero because PWL is built over a degenerate domain
    with positive-width bracket — we still allow up to 1e-2 absolute error).
    """
    if not HAS_GUROBI: return []
    results = []
    for s in range(n_samples):
        m = make_tiny_vit(seed=100+s)
        x = make_tiny_input(seed=200+s)
        with torch.no_grad():
            logits = m(torch.from_numpy(x[None]).float())[0].numpy()
        y = int(np.argmax(logits))
        true_margin = float(min(logits[y] - logits[c] for c in range(TINY_CFG['num_classes']) if c != y))
        # MILP at ε=tiny (use 1e-5 not 0 to keep PWL domains non-degenerate)
        eps = 1e-5
        verdict = verify_vit_milp(m, x, eps, y, n_pieces=n_pieces, time_limit=180)
        results.append(dict(
            sample=s, true_label=y, true_margin=true_margin,
            milp_worst_margin=verdict.get('worst_margin', None),
            verdict=verdict.get('status'),
            margin_match=(abs(verdict.get('worst_margin', np.nan) - true_margin) < 0.05),
        ))
    return results

# ── Phase 4.7 Test 2: counterexample validity ───────────────────────────
def test_counterexample_validity(n_samples=4, n_pieces=8):
    """
    Run with a generous ε that should falsify the tiny ViT.  Whenever
    status == falsified, the returned counterexample image must actually
    be misclassified by the network (PyTorch forward).  If not, the MILP
    encoding is unsound.
    """
    if not HAS_GUROBI: return []
    results = []
    for s in range(n_samples):
        m = make_tiny_vit(seed=300+s)
        x = make_tiny_input(seed=400+s)
        with torch.no_grad():
            y = int(m(torch.from_numpy(x[None]).float()).argmax(dim=-1).item())
        verdict = verify_vit_milp(m, x, eps=0.3, true_label=y,
                                  n_pieces=n_pieces, time_limit=180)
        ce_valid = None
        if verdict.get('status') == 'falsified':
            ce = verdict['counterexample']
            with torch.no_grad():
                ce_pred = int(m(torch.from_numpy(ce[None]).float()).argmax(dim=-1).item())
            ce_valid = (ce_pred != y)
        results.append(dict(sample=s, true_label=y, verdict=verdict.get('status'),
                            ce_valid=ce_valid,
                            milp_worst_margin=verdict.get('worst_margin')))
    return results

# ── Phase 4.7 Test 3: soundness vs IBP ──────────────────────────────────
def test_soundness_vs_ibp(n_samples=4, eps=0.05, n_pieces=8):
    """
    Whenever IBP verifies a sample, MILP must also verify it (MILP ≥ IBP
    in tightness for sound encodings).  An IBP-verified sample that is
    MILP-falsified indicates a bug.
    """
    if not HAS_GUROBI: return []
    results = []
    for s in range(n_samples):
        m = make_tiny_vit(seed=500+s)
        x = make_tiny_input(seed=600+s)
        with torch.no_grad():
            y = int(m(torch.from_numpy(x[None]).float()).argmax(dim=-1).item())
        # IBP
        img_lo = np.clip(x[None] - eps, 0, 1); img_up = np.clip(x[None] + eps, 0, 1)
        log_l, log_u, _ = ibp_vit(m, img_lo, img_up)
        ibp_lb_y = log_l[0, y]; ibp_ub_other = max(log_u[0, c] for c in range(TINY_CFG['num_classes']) if c != y)
        ibp_verified = (ibp_lb_y > ibp_ub_other)
        # MILP
        milp = verify_vit_milp(m, x, eps, y, n_pieces=n_pieces, time_limit=180)
        invariant_ok = (not ibp_verified) or (milp.get('status') == 'verified')
        results.append(dict(sample=s, true_label=y, ibp_verified=bool(ibp_verified),
                            milp_status=milp.get('status'),
                            invariant_ok=invariant_ok))
    return results

In [13]:
# ── Run Phase 4.7 ────────────────────────────────────────────────────────
print('═══ Test 1: identity at ε ≈ 0 ═══════════════════════════════')
r1 = test_identity_at_eps0(n_samples=3, n_pieces=8)
for x in r1:
    print(f'  sample {x["sample"]}  y={x["true_label"]}  '
          f'true_margin={x["true_margin"]:.4f}  '
          f'milp_margin={x["milp_worst_margin"]}  '
          f'match={x["margin_match"]}  ({x["verdict"]})')

print('\n═══ Test 2: counterexample validity ════════════════════════')
r2 = test_counterexample_validity(n_samples=3, n_pieces=8)
for x in r2:
    print(f'  sample {x["sample"]}  y={x["true_label"]}  verdict={x["verdict"]}  '
          f'ce_valid={x["ce_valid"]}  margin={x["milp_worst_margin"]}')

print('\n═══ Test 3: soundness vs IBP ═══════════════════════════════')
r3 = test_soundness_vs_ibp(n_samples=3, eps=0.05, n_pieces=8)
for x in r3:
    print(f'  sample {x["sample"]}  y={x["true_label"]}  '
          f'IBP={x["ibp_verified"]}  MILP={x["milp_status"]}  '
          f'invariant_OK={x["invariant_ok"]}')

# Summary
identity_ok = all(x.get('margin_match') for x in r1) if r1 else False
ce_ok = all(x.get('ce_valid') is not False for x in r2) if r2 else True
inv_ok = all(x.get('invariant_ok') for x in r3) if r3 else True
print(f'\nSummary: identity={identity_ok}  ce_valid={ce_ok}  IBP→MILP={inv_ok}')

═══ Test 1: identity at ε ≈ 0 ═══════════════════════════════


  sample 0  y=1  true_margin=0.0986  milp_margin=None  match=False  (inconclusive)
  sample 1  y=2  true_margin=0.0045  milp_margin=None  match=False  (inconclusive)
  sample 2  y=0  true_margin=0.4608  milp_margin=None  match=False  (inconclusive)

═══ Test 2: counterexample validity ════════════════════════
  sample 0  y=0  verdict=inconclusive  ce_valid=None  margin=None
  sample 1  y=0  verdict=inconclusive  ce_valid=None  margin=None
  sample 2  y=1  verdict=falsified  ce_valid=True  margin=-298.7780268743753

═══ Test 3: soundness vs IBP ═══════════════════════════════
  sample 0  y=0  IBP=False  MILP=falsified  invariant_OK=True
  sample 1  y=1  IBP=False  MILP=inconclusive  invariant_OK=True
  sample 2  y=1  IBP=False  MILP=falsified  invariant_OK=True

Summary: identity=False  ce_valid=True  IBP→MILP=True


In [14]:
# ── Save report ─────────────────────────────────────────────────────────
out_dir = Path('results/vit_p2'); out_dir.mkdir(parents=True, exist_ok=True)
report = dict(
    test1_identity_at_eps0    = r1,
    test2_counterexample_valid = r2,
    test3_soundness_vs_ibp     = r3,
)
report_path = out_dir / 'milp_vit_correctness.json'
report_path.write_text(json.dumps(report, indent=2, default=lambda x: float(x) if isinstance(x, np.generic) else None))
print(f'Saved → {report_path}')

try:
    drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
    drive_out.mkdir(parents=True, exist_ok=True)
    (drive_out / 'milp_vit_correctness.json').write_text(report_path.read_text())
    print(f'Saved drive copy → {drive_out / "milp_vit_correctness.json"}')
except Exception as e:
    print(f'(skipping drive mirror: {e})')

print('\nPhase 4 complete: MILP encoding for full ViT verified on tiny ViT.')
print('Next: Phase 6 (hybrid verifier; notebook 18).')

Saved → results/vit_p2/milp_vit_correctness.json
Saved drive copy → /content/drive/My Drive/thesis-formal-verification/results/vit_p2/milp_vit_correctness.json

Phase 4 complete: MILP encoding for full ViT verified on tiny ViT.
Next: Phase 6 (hybrid verifier; notebook 18).
